In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV

from scipy.stats import chi2_contingency

from funcs import *

set_config(transform_output="pandas")
pd.set_option('future.no_silent_downcasting', True)

# Cargar Datos

In [2]:
url = 'https://raw.githubusercontent.com/AitzolSa/dataset/619ecb7ffa2bdfcfb5a589fb8266bcf0db15250d/archive.zip'

print("Descargando y leyendo el dataset...")
X = pd.read_csv(url, compression='zip')

print("Datos cargados con exito:")
X.head()

Descargando y leyendo el dataset...
Datos cargados con exito:


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


# Dividir Train, Val y Test

In [3]:
# Dividimos train, test y validacion
semilla = 69

y = X.fraudulent
X_features = X.drop('fraudulent', axis=1)
X_resto, X_test, y_resto, y_test = model_selection.train_test_split(
    X_features, y, random_state=69, stratify=y, test_size=0.1
)
X_train, X_val, y_train, y_val = model_selection.train_test_split(
    X_resto, y_resto, random_state=69, stratify=y_resto, test_size=0.2
)

print("Train:")
print(X_train.describe())
print()

print("Val:")
print(X_val.describe())
print()

print("Test:")
print(X_test.describe())

Train:
             job_id  telecommuting  has_company_logo  has_questions
count  12873.000000   12873.000000      12873.000000   12873.000000
mean    8919.432533       0.042880          0.796784       0.492737
std     5167.644356       0.202595          0.402408       0.499967
min        2.000000       0.000000          0.000000       0.000000
25%     4408.000000       0.000000          1.000000       0.000000
50%     8935.000000       0.000000          1.000000       0.000000
75%    13380.000000       0.000000          1.000000       1.000000
max    17880.000000       1.000000          1.000000       1.000000

Val:
             job_id  telecommuting  has_company_logo  has_questions
count   3219.000000    3219.000000       3219.000000    3219.000000
mean    8991.802423       0.042249          0.788444       0.483691
std     5157.263804       0.201188          0.408475       0.499812
min        1.000000       0.000000          0.000000       0.000000
25%     4594.500000       0.000000 

# Preprocesamientos a programar

## Contar palabras

In [4]:
class contar_palabras(BaseEstimator, TransformerMixin):    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        
        X_df = X_df.apply(lambda col: col.str.split(" ").str.len())
               
        return X_df


## Contar letras

In [5]:
class contar_letras(BaseEstimator, TransformerMixin):    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        X_df = X_df.apply(lambda col: col.str.len())
               
        return X_df


## Salary range diferencias rellenado - no rellenado (1 - 0)

In [6]:

class TransformadorSalarioBinario(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        
        if 'salary_range' in X_df.columns:
            X_df['salary_range'] = X_df['salary_range'].notnull().astype(int)
        
        return X_df

## Salary range alto-medio-bajo-vacio

In [7]:
class discretizador_salario(BaseEstimator, TransformerMixin):   
    def separar(self,string):
        
        if pd.isna(string):
            return "null"
            
        string_strip = str(string).strip()
            
        if "-" not in string_strip:
            try:
                salary = int(string_strip)
            except:
                return "null"
                
        else:
            try:
                salary_range = string_strip.split("-")
                
                salary = (float(salary_range[0]) + float(salary_range[1])) / 2
            except ValueError:
                return "null"

        if salary < 50000:
            return "low"
        if salary > 120000:
            return "high"
        return "medium"            
        
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        
        for col in X_df.columns: # Esto no deberia ser un problema porque solo se lo vamos a aplicar a 1 columna
            X_df.loc[:,col] = X_df.loc[:,col].apply(self.separar)
               
        return X_df


## Eliminar ejemplos que no salgan mucho
    
    Este cuidao, hacemos un par de pruebas y a ver qtal va, que puede quitar información y no merecer la pena

In [8]:

class FiltroFrecuenciaCategorica(BaseEstimator, TransformerMixin):
    def __init__(self, column, minimo=10):
        self.column = column
        self.minimo = minimo
        self.valid_categories_ = None

    def fit(self, X, y=None):
        counts = X[self.column].value_counts()
        self.valid_categories_ = counts[counts >= self.minimo].index.tolist()
        return self

    def transform(self, X):
        X_df = X.copy()
        X_df[self.column] = X_df[self.column].where(X_df[self.column].isin(self.valid_categories_), np.nan)
        
        return X_df

## Binary Encoding

In [16]:
class BinaryEncoder:
    
    def __init__(self):
        self.mapeos = {}
        self.num_bits = {}

    def fit(self, X, columnas):
        for col in columnas:
            categorias = X[col].dropna().unique()
            mapa = {}
            
            for i, cat in enumerate(categorias):
                mapa[cat] = i
            
            self.mapeos[col] = mapa
            n_categorias = len(categorias)
            
            if n_categorias > 1:
                bits = math.ceil(math.log2(n_categorias))
            else:
                bits = 1
            
            self.num_bits[col] = bits
        return self

    def transform(self, X):
        
        X_transformado = X.copy()

        for col in self.mapeos:
            mapa = self.mapeos[col]
            bits = self.num_bits[col]
            valores_numericos = X_transformado[col].map(mapa)
            valores_numericos = valores_numericos.fillna(0)
            valores_numericos = valores_numericos.astype(int)
            
            for i in range(bits):
                nueva_col = []
                
                for x in valores_numericos:
                    valor = (x >> i) & 1
                    nueva_col.append(valor)
                X_transformado[col + "_bin_" + str(i)] = nueva_col
            X_transformado = X_transformado.drop(columns=[col])

        return X_transformado

    def fit_transform(self, X, columnas):
        self.fit(X, columnas)
        return self.transform(X)

## Extra 1

## Extra 2

## Extra 3